In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [6]:
def rebuild_abstract(indexed_abstract):
    if not indexed_abstract or "InvertedIndex" not in indexed_abstract:
        return ""
    
    length = indexed_abstract.get("IndexLength", 0)
    inverted_index = indexed_abstract["InvertedIndex"]
    
    # Create an array of empty slots for words
    words = [""] * length
    
    # Place each token at its respective positions
    for token, positions in inverted_index.items():
        for pos in positions:
            if pos < length:
                words[pos] = token
                
    return " ".join(words).strip()

In [ ]:
import ijson
import time
import csv
import numpy as np
from decimal import Decimal
from tqdm import tqdm
start = time.process_time()

PAPER = []
Author = []
count = 0

with open('/Users/iamgeorgerieh/Downloads/dblp.v12.json', "rb") as f, open("output.csv", "w", newline="") as csvfile:
    fieldnames = ['id', 
                  'title', 
                  'year', 
                  "author_list",
                  'doc_type',
                  'references',
                'venue_list',
                  'doi', 
                  'keyword',
                  'publisher',
                  'rebuilt_abstract', 
                  ]
    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
    writer.writeheader()
    with tqdm(total=4894081) as pbar:
        for i, element in tqdm(enumerate(ijson.items(f, "item"))):
            if element['doc_type'] != 'Journal':
                pass
            else:
                paper = {}
                paper['id'] = element['id']
                paper['title'] = element['title']

                year = element.get('year')
                if year:
                    paper['year'] = year
                else:
                    paper['year'] = ""

                author = element.get('authors')
                if author:
                    Author = element['authors']
                    author_list = []


                    for i in Author:
                        author_list.append((i.get("name"), i.get("id"), i.get("org")))
                    paper['author_list'] = author_list

                doc_type = element.get('doc_type')
                if doc_type:
                    paper['doc_type'] = doc_type
                else:
                    paper['doc_type'] = ""

                references = element.get('references')
                if references:
                    paper['references'] = ';'.join(str(int(r)) for r in references)
                else:
                    paper['references'] = ""

                i = element.get('venue')
                paper['venue_list'] = (i.get('id'), i.get('raw'), i.get('type'))
                doi = element.get('doi')
                if doi:
                    paper['doi'] = f"https://doi.org/{doi}"
                else:
                    paper['doi'] = ""

                fos = element.get('fos')
                if fos:
                    fosunparsed = element['fos']
                    keyword = []
                    # weight = []

                    for i in fosunparsed:
                        if isinstance(i['w'], (int, float, Decimal)):
                            keyword.append(str(i['name']))  # Convert to string

                else:
                    keyword = []

                paper['keyword'] = ';'.join(keyword)
                paper['rebuilt_abstract'] = rebuild_abstract(element.get('indexed_abstract')) 
                publisher= element.get('publisher')
                if publisher:
                    paper['publisher']=publisher
                else:
                    paper['publisher']=""
                writer.writerow(paper)
            pbar.update(1)


4894081it [07:57, 10241.28it/s]1 [07:57<00:00, 18910.86it/s]
100%|██████████| 4894081/4894081 [07:57<00:00, 10241.19it/s]


**Creation of `dataframe` from python list of objects ->**

In [16]:
import pandas as pd

In [17]:
df = pd.read_csv('output.csv')

NameError: name 'df' is not defined